In [ ]:
import hashlib, json, os, shutil, subprocess, sys
from pathlib import Path
def _d1_parse_phase(raw):
    value=(raw or "").strip().lower()
    if value in ("", "0", "false", "no"): return "FAST_SAVE"
    if value in ("1", "true", "yes"): return "FULL_RERUN"
    raise RuntimeError(f"unexpected KAGGLE_IS_COMPETITION_RERUN value: {raw!r}")
def _d1_placeholder_payload(mounted):
    if not isinstance(mounted,dict) or not mounted: raise RuntimeError("mounted competition challenge is empty or invalid")
    payload={}
    for task_id,task in mounted.items():
        tests=task.get("test") if isinstance(task,dict) else None
        if not isinstance(tests,list) or not tests: raise RuntimeError(f"invalid test structure for {task_id}")
        payload[task_id]=[{"attempt_1":[[0]],"attempt_2":[[0]]} for _ in tests]
    return payload
def _d1_atomic_json(path,payload):
    path.parent.mkdir(parents=True,exist_ok=True); tmp=path.with_name(path.name+".tmp")
    tmp.write_text(json.dumps(payload,sort_keys=True)+"\n",encoding="utf-8"); os.replace(tmp,path)
def _d1_fast_save(challenge,submission,provenance):
    mounted=json.loads(challenge.read_text(encoding="utf-8")); payload=_d1_placeholder_payload(mounted)
    _d1_atomic_json(submission,payload)
    record={"phase":"FAST_SAVE_ONLY","model_loaded":False,"ttt_executed":False,"full_inference_executed":False,"submission_kind":"SAVE_ONLY_PLACEHOLDER","task_count":len(payload),"test_output_count":sum(len(v) for v in payload.values()),"challenge_sha256":hashlib.sha256(challenge.read_bytes()).hexdigest(),"submission_sha256":hashlib.sha256(submission.read_bytes()).hexdigest()}
    _d1_atomic_json(provenance,record); print(json.dumps({"event":"D1_FAST_SAVE_COMPLETE",**record},sort_keys=True),flush=True)
def _d1_full_rerun(challenge,submission,fast_provenance):
    submission.unlink(missing_ok=True); fast_provenance.unlink(missing_ok=True)
    work=Path("/kaggle/working"); source_archive=Path("/kaggle/input/datasets/jimmy5566/arc2-d1-release-source/ARC2.tar")
    expected_source_sha256="2b4bc0043bb96e400bbbee77eccffb6168250b36ed0af0721b9816737d4642f9"
    source_root=source_archive.parent/"ARC2"; source_manifest=json.loads((source_archive.parent/"SOURCE_MANIFEST.json").read_text())
    if source_manifest["archive_sha256"]!=expected_source_sha256 or not source_root.is_dir(): raise RuntimeError("explicit D1 source identity/mount mismatch")
    expected_files=source_manifest["file_sha256"]; mounted_files={Path(base,name).relative_to(source_root).as_posix() for base,_,files in os.walk(source_root) for name in files}
    if mounted_files!=set(expected_files): raise RuntimeError("D1 mounted source file set mismatch")
    for name,expected in expected_files.items():
        if hashlib.sha256((source_root/name).read_bytes()).hexdigest()!=expected: raise RuntimeError(f"D1 mounted source hash mismatch: {name}")
    root=work/"ARC2"
    if root.exists(): shutil.rmtree(root)
    shutil.copytree(source_root,root)
    config=Path("/kaggle/input/datasets/jimmy5566/arc2-d1-release-source/d1_release_config.json")
    if not config.is_file(): raise RuntimeError("explicit D1 release config missing")
    if hashlib.sha256(config.read_bytes()).hexdigest()!=source_manifest["config_sha256"]: raise RuntimeError("D1 release config hash mismatch")
    cfg=json.loads(config.read_text())
    if cfg["environment"].get("bootstrap_mode")!="pinned_kaggle_image_offline": raise RuntimeError("unverified offline bootstrap mode")
    os.environ.update({"TRITON_PTXAS_PATH":cfg["environment"]["ptxas_path"],"HF_HUB_OFFLINE":"1","TRANSFORMERS_OFFLINE":"1","TOKENIZERS_PARALLELISM":"false"})
    print(json.dumps({"event":"D1_OFFLINE_REFERENCE_ENV_BOOTSTRAPPED","mode":"pinned_kaggle_image_offline","ptxas":os.environ["TRITON_PTXAS_PATH"]},sort_keys=True),flush=True)
    import importlib.metadata as md
    if not sys.version.startswith(cfg["environment"]["python_prefix"]): raise RuntimeError(f"Python mismatch: {sys.version}")
    for package in ("unsloth","unsloth-zoo","transformers","torch","torchao","peft","trl","triton"):
        if md.version(package)!=cfg["environment"][package]: raise RuntimeError(f"frozen dependency mismatch: {package}")
    ptxas=Path(cfg["environment"]["ptxas_path"]); model=Path("/kaggle/input/models/sorokin/qwen3_4b_grids15_sft139/transformers/bfloat16/1")
    if not ptxas.is_file() or subprocess.run([str(ptxas),"--version"],capture_output=True).returncode: raise RuntimeError("verified ptxas unavailable")
    if not model.is_dir(): raise RuntimeError("explicit model mount missing")
    out=work/"artifacts"/"d1_release"; out.mkdir(parents=True,exist_ok=True); candidates=out/"candidates_frozen.json"; selection=out/"d1_selection_frozen.json"; provenance=out/"PRODUCTION_PROVENANCE.json"
    print(json.dumps({"event":"D1_RELEASE_FULL_RERUN","challenge":str(challenge),"source_sha256":expected_source_sha256,"model":str(model),"portfolio":cfg["generation"]["portfolio"]},sort_keys=True),flush=True)
    run=[sys.executable,str(root/"scripts"/"run_d1_release_4gpu.py"),"--challenge",str(challenge),"--release-config",str(config),"--model-path",str(model),"--native-config-dir",str(root/"configs"/"nvarc_native_846d0198"),"--checkpoint-dir",str(out/"checkpoints"),"--output",str(candidates),"--resume"]
    if subprocess.run(run,env={**os.environ,"TRITON_PTXAS_PATH":str(ptxas),"HF_HUB_OFFLINE":"1","TRANSFORMERS_OFFLINE":"1"}).returncode: raise RuntimeError("D1_REAL_WORKERS_FAILED")
    finalize=[sys.executable,str(root/"scripts"/"build_d1_release_submission.py"),"--challenge",str(challenge),"--release-config",str(config),"--records",str(candidates),"--selection-output",str(selection),"--provenance-output",str(provenance),"--output",str(submission)]
    if subprocess.run(finalize,env={**os.environ,"CUDA_VISIBLE_DEVICES":""}).returncode: raise RuntimeError("D1_PER_OUTPUT_FINALIZATION_FAILED")
    if not submission.is_file(): raise RuntimeError("D1 submission missing")
    payload=json.loads(submission.read_text()); mounted=json.loads(challenge.read_text())
    if set(payload)!=set(mounted) or any(len(payload[k])!=len(mounted[k]["test"]) for k in mounted): raise RuntimeError("runtime challenge/submission mapping mismatch")
    print(json.dumps({"event":"D1_RELEASE_COMPLETE","task_count":len(payload),"test_output_count":sum(len(v) for v in payload.values()),"submission_sha256":hashlib.sha256(submission.read_bytes()).hexdigest(),"solutions_opened":False},sort_keys=True),flush=True)
def _d1_route(raw,fast_save_callable,full_rerun_callable):
    phase=_d1_parse_phase(raw)
    if phase=="FAST_SAVE": fast_save_callable()
    else: full_rerun_callable()
    return phase
def _d1_main():
    work=Path("/kaggle/working"); challenge=Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json")
    if not challenge.is_file(): raise RuntimeError("mounted competition challenge missing")
    submission=work/"submission.json"; fast_provenance=work/"artifacts"/"d1_release"/"FAST_SAVE_PROVENANCE.json"
    return _d1_route(os.getenv("KAGGLE_IS_COMPETITION_RERUN", ""),lambda:_d1_fast_save(challenge,submission,fast_provenance),lambda:_d1_full_rerun(challenge,submission,fast_provenance))
_d1_main()
